## TIFON Databases. Read, load and merge two CODA databases.

In [1]:
import sys
try:
    import pyLOM
except ImportError as e:
    print(f'Error importing pyLOM: {e}')
    print('Importing with local repository')
    sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM

# try:
#     import shutil as sh
#     sh.rmtree('/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_completed', ignore_errors=True)
    
# except Exception as e:
#     print(f"Error removing directory: {e}")

def read_db(datafolder, case_idx):
    db = FRODO(root_dir = datafolder, format = 'CODA', initial_parse = True)
    
    db.extract_inputs(
        id_groups = (3,),
        cases_idx = case_idx,
        vtu_type='surface',
        verbose=False
        )

    db.extract_outputs(
        id_groups=(3,),
        stage=0, cases_idx = case_idx,
        var_name_excluded = [
            'BoundaryValues_CoefSkinFrictionX',
            'BoundaryValues_CoefSkinFrictionY',
            'BoundaryValues_CoefSkinFrictionZ'
            ],
        vtu_type='surface',
        )

    db.extract_outputs(
        id_groups=(3,),
        stage=1, cases_idx = case_idx,
        var_name_excluded = [
            'BoundaryValues_CoefSkinFrictionX',
            'BoundaryValues_CoefSkinFrictionY',
            'BoundaryValues_CoefSkinFrictionZ'
            ],
        vtu_type='surface',
        )
    
    return db

Error importing pyLOM: No module named 'pyLOM'
Importing with local repository


0 Warning! Import - NVTX not present!
/home/m.jaraiz/miniconda/envs/envkan_amd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Base de datos original
case_idx = list(range(100))
fuera = [64, 79, 87, 88, 94]
for c in fuera:
    case_idx.remove(c)
    
# case_idx = list(range(10))
db_0 = read_db(datafolder='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3/', case_idx=case_idx)

In [ ]:
# Base de datos adicional
db_1 = read_db('/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_prueba/', case_idx='all')

In [ ]:
print(db_0.metadata['design_vars'])
print(db_1.metadata['design_vars'])
flcc = db_1.data_dict['CADGroup_3']['FlCc'][:, :-1]
design_vars = db_1.metadata['design_vars']
db_1.metadata['design_vars'] = design_vars[:-1]
db_1.data_dict['CADGroup_3']['FlCc'] = flcc
print(db_0.metadata['design_vars'])
print(db_1.metadata['design_vars'])

In [ ]:
for db in [db_0, db_1]:
    print(db.data_dict['CADGroup_3']['FlCc'].shape)

### Merge two sets

In [ ]:
db_completo = FRODO.merge_datasets(
    root_dir='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_completed',
    sources = [(db_0, '3'), (db_1, '3')],
    new_group_id='3_completo',
    k=4,
    mesh_ref=0,
    cache=True,
    get_df_metrics_attr={
        'var_metrics': ['CoefLift', 'CoefDrag', 'CoefMomentY'],
        'iter_var': 1000,
        'save' : False
    }
    
)
db_completo.summary_data()

In [ ]:
import pandas as pd
df_post = pd.read_csv(
    '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_completed/metadata/df_post.csv',
    sep=',',
    header=0,
)
# columnas_cambiar = [c for c in df_post.columns if c.startswith('coef')]

# mask = df_post['folder'] == 'aoa_4.7097_mach_0.7784_h_11000' # mask = df_post['dataset'] == 'dataset_1'
# folders = df_post.loc[mask, 'folder']

# df_post.loc[mask, columnas_cambiar] *= 0.1   # Stage [0, 1] - 

# columnas_cambiar = [c for c in df_post.columns if all((c.startswith('coef'), c.endswith('stage0')))]
# df_post.loc[mask, columnas_cambiar] *= 0.1
# df_post.sort_values(by=['aoa', 'mach'], inplace=True)

In [ ]:
def corregir_valores_coef(root_dir, folder_names, stage, factor):
    import numpy as np
    import os
    
    for folder in folder_names:
        print(f'\nCaso: {folder}')
        file = f"output_{stage}__monitors_wall_boundary_integrals.dat"      
        with open(os.path.join(root_dir, 'outputs', folder, file), "r") as f:
            header1 = f.readline()
            header2 = f.readline()
            data_lines = f.readlines()
            
        data = np.loadtxt(data_lines)
        # VARIABLES = "CoefDrag" "CoefLift" "CoefMomentY" "Iteration" "Time"
        data[:, 0:3] *= factor
        
        with open(os.path.join(root_dir, 'outputs', folder, file), "w") as f:
            f.write(header1)
            f.write(header2)
            np.savetxt(f, data, fmt="%.15e")

mask = df_post['dataset'] == 'dataset_0'
folders = df_post.loc[mask, 'folder']  
corregir_valores_coef(root_dir = db_0.root_dir, folder_names=folders.to_list(), stage=0, factor=10)

mask = df_post['dataset'] == 'dataset_1'
folders = df_post.loc[mask, 'folder']  
corregir_valores_coef(root_dir = db_1.root_dir, folder_names=folders.to_list(), stage=0, factor=10)
# corregir_valores_coef(root_dir = db_1.root_dir, folder_names=folders, stage=1, factor=0.1)
# corregir_valores_coef(root_dir = db_1.root_dir, folder_names=folders, stage=0, factor=0.1)

In [ ]:
db_completo.plot_state()

In [ ]:
from FotR import LEGOLAS
theElf = LEGOLAS(db=db_completo)

theElf.params.plot_space()

In [ ]:
columns0 = [c for c in df_post.columns.tolist() if all(('stage0' in c, c.startswith('coef')))]
columns1 = [c for c in df_post.columns.tolist() if all(('stage1' in c, c.startswith('coef')))]

import plotly.express as px
for c0, c1 in zip(columns0, columns1):
    fig = px.scatter(
        df_post,
        x=c0,
        y=c1,
        color='aoa',
        hover_data=['case_idx', 'mach', 'dataset']
    )
    fig.show()


In [ ]:
import plotly.express as px
fig = px.scatter(
    df_post,
    x='coeflift_mean_stage0',
    y='coeflift_mean_stage1',
    color='aoa',
    hover_data=['case_idx', 'mach', 'dataset']
)
fig.show()

import plotly.express as px
fig = px.scatter(
    df_post,
    x='coefdrag_mean_stage0',
    y='coefdrag_mean_stage1',
    color='aoa',
    hover_data=['case_idx', 'mach', 'dataset']
)
fig.show()

In [ ]:
fig = px.scatter(
    df_post,
    x="aoa",
    y="coeflift_mean_stage1",
    color="mach",
    title="Polar CL vs AoA"
)
fig.show()

In [ ]:
fig = px.scatter(
    df_post,
    x="coefdrag_mean_stage1",
    y="coeflift_mean_stage1",
    color="aoa"
)
fig.show()

In [ ]:
d = []
for stage, factor in zip([0, 1], [1, 1]):
    vars = {
        'aoa' : {'idim':0, 'value':df_post['aoa'].values}, # Angle of attack
        'M'   : {'idim':0, 'value':df_post['mach'].values},   # Mach number
        'Re'  : {'idim':0, 'value':df_post['re'].values},  # Reynolds
        'CL'  : {'idim':0, 'value':df_post[f'coeflift_mean_stage{stage}'].values / factor},  # Mean Lift coefficient
        'CD'  : {'idim':0, 'value':df_post[f'coefdrag_mean_stage{stage}'].values / factor},  # Mean Drag coefficient
        'CMy'  : {'idim':0, 'value':df_post[f'coefmomenty_mean_stage{stage}'].values / factor},  # Mean Momentum coefficient
        'varCL'  : {'idim':0, 'value':df_post[f'coeflift_var_stage{stage}'].values},  # Var Lift coefficient
        'varCD'  : {'idim':0, 'value':df_post[f'coefdrag_var_stage{stage}'].values},  # Var Drag coefficient
        'varCM'  : {'idim':0, 'value':df_post[f'coefmomenty_var_stage{stage}'].values},  # Var Momentum coefficient
    }
    d.append(db_completo.sets.create_NN_pylom(id_groups=['3_completo',], stage=stage, idx_to_print='all', external_vars=vars, save_path='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_completed/outputs/'))

### Interpolation between meshes

In [ ]:
new_mesh = {
    "Coord": db_1.data_dict['CADGroup_3']['Coord'],
    "Conec": db_1.data_dict['CADGroup_3']['Conec'],
    "NodeCoord": db_1.data_dict['CADGroup_3']['NodeCoord'],
    # lo que tengas
}
db_0.sets.create_group_by_interpolation(
    id_group_src='3',
    new_group_id='3_interp',
    new_mesh=new_mesh,
    method='idw',
    k=8
    )

In [ ]:
db_0.summary_data()